# E-Commerce Analytics Project
## 05 — Dashboard Dataset Preparation

### Objective
Create a final dashboard-ready analytical dataset by combining the cleaned SQL transaction view with customer segmentation and dashboard-friendly date fields.

The resulting dataset will support the executive BI dashboard.

In [1]:
import pandas as pd
import numpy as np
import sqlite3

from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATABASE_PATH = (
    PROJECT_ROOT / "database" / "ecommerce.db"
)

ANALYSIS_DIR = (
    PROJECT_ROOT / "data" / "analysis"
)

DASHBOARD_DIR = (
    PROJECT_ROOT / "dashboard"
)

DASHBOARD_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
connection = sqlite3.connect(
    DATABASE_PATH
)

dashboard_df = pd.read_sql_query(
    """
    SELECT *
    FROM analytics_transactions
    WHERE order_status = 'Completed';
    """,
    connection
)

connection.close()

print(
    f"Dashboard transaction rows: "
    f"{len(dashboard_df):,}"
)

Dashboard transaction rows: 121,525


In [4]:
dashboard_df["order_date"] = pd.to_datetime(
    dashboard_df["order_date"]
)

In [5]:
dashboard_df["year"] = (
    dashboard_df["order_date"].dt.year
)

dashboard_df["quarter"] = (
    "Q"
    + dashboard_df[
        "order_date"
    ].dt.quarter.astype(str)
)

dashboard_df["month_number"] = (
    dashboard_df["order_date"].dt.month
)

dashboard_df["month_name"] = (
    dashboard_df["order_date"].dt.month_name()
)

dashboard_df["year_month"] = (
    dashboard_df["order_date"]
    .dt.to_period("M")
    .astype(str)
)

In [6]:
customer_order_counts = (
    dashboard_df
    .groupby("customer_id")["order_id"]
    .nunique()
)

customer_type_lookup = (
    customer_order_counts
    .reset_index(
        name="customer_order_count"
    )
)

customer_type_lookup["customer_type"] = np.where(
    customer_type_lookup[
        "customer_order_count"
    ] > 1,
    "Returning",
    "One-Time"
)

In [7]:
dashboard_df = dashboard_df.merge(
    customer_type_lookup,
    on="customer_id",
    how="left"
)

In [8]:
rfm = pd.read_csv(
    ANALYSIS_DIR / "rfm_customers.csv"
)

In [9]:
rfm_lookup = rfm[
    [
        "customer_id",
        "recency",
        "frequency",
        "monetary",
        "rfm_score",
        "segment"
    ]
].copy()

In [10]:
dashboard_df = dashboard_df.merge(
    rfm_lookup,
    on="customer_id",
    how="left"
)

In [11]:
dashboard_df["gross_margin_pct"] = np.where(
    dashboard_df["revenue"] != 0,
    dashboard_df["gross_profit"]
    / dashboard_df["revenue"]
    * 100,
    np.nan
)

In [12]:
dashboard_df["net_revenue_after_refunds"] = (
    dashboard_df["revenue_after_refunds"]
)

In [13]:
dashboard_df["return_status"] = np.where(
    dashboard_df["was_returned"] == 1,
    "Returned",
    "Not Returned"
)

In [14]:
print(
    "Rows:",
    f"{len(dashboard_df):,}"
)

print(
    "Columns:",
    dashboard_df.shape[1]
)

print(
    "Duplicate order item IDs:",
    dashboard_df[
        "order_item_id"
    ].duplicated().sum()
)

print(
    "Missing RFM segments:",
    dashboard_df[
        "segment"
    ].isna().sum()
)

Rows: 121,525
Columns: 43
Duplicate order item IDs: 0
Missing RFM segments: 0


In [15]:
dashboard_df.head()

,order_item_id,order_id,order_date,customer_id,city,state,region,age_group,acquisition_channel,device,...,customer_order_count,customer_type,recency,frequency,monetary,rfm_score,segment,gross_margin_pct,net_revenue_after_refunds,return_status
0,ITEM-0000001,ORD-000001,2025-03-05,CUST-20202,Port Gabriel,TX,South,25-34,Affiliate,Desktop,...,6,Returning,83,6,1176.2455,12,Regular,5.684836,158.14,Not Returned
1,ITEM-0000002,ORD-000001,2025-03-05,CUST-20202,Port Gabriel,TX,South,25-34,Affiliate,Desktop,...,6,Returning,83,6,1176.2455,12,Regular,43.055054,221.60,Not Returned
2,ITEM-0000003,ORD-000001,2025-03-05,CUST-20202,Port Gabriel,TX,South,25-34,Affiliate,Desktop,...,6,Returning,83,6,1176.2455,12,Regular,26.619628,31.18,Not Returned
3,ITEM-0000004,ORD-000002,2024-12-05,CUST-18773,Davidfurt,PA,Northeast,45-54,Social Media,Desktop,...,2,Returning,351,2,774.5700,5,Low Engagement,38.649554,211.93,Not Returned
4,ITEM-0000005,ORD-000002,2024-12-05,CUST-18773,Davidfurt,PA,Northeast,45-54,Social Media,Desktop,...,2,Returning,351,2,774.5700,5,Low Engagement,43.540873,0.00,Returned


In [16]:
dashboard_kpis = {
    "Revenue":
        dashboard_df["revenue"].sum(),

    "Revenue After Refunds":
        dashboard_df[
            "net_revenue_after_refunds"
        ].sum(),

    "Gross Profit":
        dashboard_df["gross_profit"].sum(),

    "Orders":
        dashboard_df["order_id"].nunique(),

    "Customers":
        dashboard_df[
            "customer_id"
        ].nunique(),

    "Units Sold":
        dashboard_df["quantity"].sum(),

    "Returned Items":
        dashboard_df["was_returned"].sum()
}

for metric, value in dashboard_kpis.items():
    print(
        f"{metric:<25} "
        f"{value:,.2f}"
    )

Revenue                   26,495,888.73
Revenue After Refunds     24,372,291.53
Gross Profit              11,293,660.61
Orders                    72,756.00
Customers                 19,521.00
Units Sold                168,301.00
Returned Items            10,269.00


In [17]:
dashboard_file = (
    DASHBOARD_DIR
    / "ecommerce_dashboard_data.csv"
)

dashboard_df.to_csv(
    dashboard_file,
    index=False
)

print(
    "Dashboard dataset exported:"
)

print(dashboard_file)

Dashboard dataset exported:
/Users/gonzalojenkinsmadrid/E-Commerce-Analytics/dashboard/ecommerce_dashboard_data.csv


In [18]:
marketing_dashboard = pd.read_csv(
    ANALYSIS_DIR
    / "marketing_efficiency.csv"
)

marketing_dashboard.to_csv(
    DASHBOARD_DIR
    / "marketing_dashboard_data.csv",
    index=False
)

print(
    "Marketing dashboard dataset exported."
)

Marketing dashboard dataset exported.


## Conclusion

A final BI-ready analytical dataset was created from the SQL transaction view.

The dataset includes:
- Transaction and order information
- Customer and geographic attributes
- Product and category information
- Revenue, COGS, and gross profit
- Returns and refunds
- Customer type
- RFM customer segmentation
- Dashboard-ready date dimensions

Marketing efficiency data was preserved as a separate analytical dataset to avoid duplicating channel-level spend across transaction records.

The project data is now ready for dashboard development.